In [ ]:
# Cell 1 — Setup
"""
05_baselines.ipynb
==================
Interactive baseline notebook focused on DirectGNN and descriptor-model comparison.

CLI equivalents and related scripts:
- `python scripts/train_directgnn.py ...`
- `python scripts/run_seeds.py --train-script scripts/train_directgnn.py ...`
- `python scripts/run_fastsolv.py ...`
- `python scripts/run_split_comparisons.py --splits "solute_scaffold,solute,solvent" ...`
- `python scripts/learning_curves.py --models "tgnn_solv,direct_gnn,rf_baseline" ...`
- `python scripts/statistical_tests.py --results ... --labels ...`
- `src/tgnn_solv/baselines/rf_baseline.py` and `ideal_sle.py` for classical baselines
"""

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"
NOTEBOOK_FIG_DIR = FIGURES_DIR / "notebooks"
NOTEBOOK_RESULTS_DIR = RESULTS_DIR / "notebooks"
NOTEBOOK_FIG_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import torch
import pandas as pd
import matplotlib.pyplot as plt

from tgnn_solv.config import TGNNSolvConfig
from tgnn_solv.inference import load_model
from tgnn_solv.data import make_loaders, PROCESSED_DIR
from tgnn_solv.baselines import run_baseline, compare_with_tgnn

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")


In [ ]:
# Cell 2 — Load data
# Baseline work is usually reported on the random-by-solute split.
# Switch back to train.csv / val.csv / test.csv if you want the strict scaffold split.
train_df = pd.read_csv(PROCESSED_DIR / "train_solute.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val_solute.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test_solute.csv")

cfg = TGNNSolvConfig(
    hidden_dim=256,
    n_gnn_layers=6,
    encoder_role_mode="shared_residual",
    encoder_role_specific_layers=2,
    n_cross_attn_layers=3,
    n_attn_heads=8,
    pair_dim=512,
    nrtl_tau_mode="ref_invT",
    use_pair_temperature_batching=True,
    pair_temperature_min_group_size=2,
    pair_temperature_group_chunk_size=4,
)

train_loader, val_loader, test_loader = make_loaders(
    train_df,
    val_df,
    test_df,
    batch_size=cfg.batch_size,
    use_pair_temperature_batching=cfg.use_pair_temperature_batching,
    pair_temperature_min_group_size=cfg.pair_temperature_min_group_size,
    pair_temperature_group_chunk_size=cfg.pair_temperature_group_chunk_size,
)


In [ ]:
# Cell 3 — Train DirectGNN baseline
baseline_metrics = run_baseline(
    train_loader, val_loader, test_loader,
    cfg=cfg, device=DEVICE,
    n_epochs=2, patience=20,
)

In [ ]:
# Cell 4 — Compare with TGNN-Solv
MODEL_PATH = CHECKPOINT_DIR / "tgnn_solv_trained.pt"
model, model_cfg = load_model(str(MODEL_PATH), DEVICE)

comparison = compare_with_tgnn(
    model, model_cfg, baseline_metrics,
    test_loader, test_df,
)

comparison.to_csv(NOTEBOOK_RESULTS_DIR / "baseline_comparison.csv", index=False)
comparison


In [ ]:
# Cell 5 — Visualization

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

names = comparison["name"].values
colors = ["coral", "steelblue"]

for ax, metric, title, better in [
    (axes[0], "mae", "MAE (ln x₂) ↓", "lower"),
    (axes[1], "rmse", "RMSE (ln x₂) ↓", "lower"),
    (axes[2], "r2", "R² ↑", "higher"),
]:
    vals = comparison[metric].values
    bars = ax.bar(names, vals, color=colors, width=0.5, edgecolor="black")
    ax.set_title(title, fontsize=12)
    ax.set_ylabel(metric.upper())

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f"{val:.3f}", ha="center", va="bottom", fontsize=11)

plt.tight_layout()
plt.savefig(NOTEBOOK_FIG_DIR / "baseline_comparison.png", dpi=150)
plt.show()


In [ ]:
# Cell 6 — Temperature encoding sanity check

from tgnn_solv.baselines.temperature import ThermometerEncoder

enc = ThermometerEncoder(n_bins=10, T_min=200, T_max=500)

test_temps = torch.tensor([250.0, 298.15, 350.0, 450.0])
encoded = enc.encode(test_temps)

print("Temperature encoding examples (10 bins, 200-500 K):")
print(f"Bin width: {enc.bin_width:.0f} K")
for i, T_val in enumerate(test_temps):
    vals = [f"{v:.2f}" for v in encoded[i].tolist()]
    print(f"  T={T_val.item():6.1f} K → [{', '.join(vals)}]")